# 8.3 Partial order planning: practice notebook

Build plans that order steps only when they must. Learn causal links, threats, promotion and demotion, and watch the partial order planner solve socks and shoes, the spare tyre and the Sussman anomaly.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Counting the linearizations of socks and shoes

In [ ]:
# A partial order plan keeps only the orderings it needs. Count the total orders (linearizations) it allows.
steps = ["RightSock", "RightShoe", "LeftSock", "LeftShoe"]
before = [("RightSock", "RightShoe"), ("LeftSock", "LeftShoe")]   # (a, b) means a must come before b

def linearize(done, left):
    if not left:
        return [done]
    out = []
    for s in left:
        ready = True
        for a, b in before:
            if b == s and a in left:      # a must come first but has not been placed
                ready = False
        if ready:
            rest = [x for x in left if x != s]
            out = out + linearize(done + [s], rest)
    return out

orders = linearize([], steps)
print("total orders allowed:", len(orders))
for o in orders:
    print("  ", ", ".join(o))

### A small partial order planner on the spare tyre

In [ ]:
# A small partial order planner: steps, orderings and causal links, with threats fixed by promotion or demotion.
acts = {"Remove(Spare, Trunk)": (["At(Spare, Trunk)"], ["At(Spare, Ground)"], ["At(Spare, Trunk)"]),
        "Remove(Flat, Axle)": (["At(Flat, Axle)"], ["At(Flat, Ground)", "Free(Axle)"], ["At(Flat, Axle)"]),
        "PutOn(Spare, Axle)": (["At(Spare, Ground)", "Free(Axle)"], ["At(Spare, Axle)"], ["At(Spare, Ground)", "Free(Axle)"])}
acts["Start"] = ([], ["At(Flat, Axle)", "At(Spare, Trunk)"], [])     # (pre, add, del)
acts["Finish"] = (["At(Spare, Axle)"], [], [])

def after(order, a, b):              # is a forced to come after b (b before a)?
    seen, todo = [], [b]
    while todo:
        x = todo.pop()
        for p, q in order:
            if p == x and q not in seen:
                seen.append(q)
                todo.append(q)
    return a in seen

def pop(plan, order, links, agenda):
    if not agenda:
        return plan, order
    p, need = agenda[0]
    for s in plan + [a for a in acts if a not in plan]:
        if p not in acts[s][1] or after(order, s, need) or s == need:
            continue
        new_plan = plan + [s] if s not in plan else plan
        new_order = order + [(s, need)]
        new_agenda = agenda[1:] + ([(q, s) for q in acts[s][0]] if s not in plan else [])
        new_links = links + [(s, p, need)]
        ok = True
        for a, f, b in new_links:            # fix every threat: a step that deletes f could run between a and b
            for t in new_plan:
                if f in acts[t][2] and t != a and t != b and not after(new_order, a, t) and not after(new_order, t, b):
                    if not after(new_order, t, a) and t != "Start":
                        new_order = new_order + [(t, a)]      # demotion: t before a
                    elif not after(new_order, b, t) and b != "Finish":
                        new_order = new_order + [(b, t)]      # promotion: t after b
                    else:
                        ok = False
        if ok:
            print("link", s, "--", p, "->", need)
            result = pop(new_plan, new_order, new_links, new_agenda)
            if result:
                return result
    return None

plan, order = pop(["Start", "Finish"], [("Start", "Finish")], [], [("At(Spare, Axle)", "Finish")])
print("steps:", [s for s in plan if s not in ("Start", "Finish")])

## Your turn

### Task 1: Check that orderings have no cycle

before is a list of pairs (a, b) meaning a ≺ b. Write `consistent(before)` that returns True if no step must come before itself, and False if the orderings have a cycle.

Hint: For every step, follow the pairs forward and see whether you come back to the same step.

In [ ]:
def consistent(before):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("consistent([('A', 'B'), ('B', 'C')])", True), ("consistent([('A', 'B'), ('B', 'A')])", False), ("consistent([('A', 'B'), ('B', 'C'), ('C', 'A')])", False), ('consistent([])', True)])

### Task 2: Fix a threat by demotion or promotion

Step t threatens the causal link from a to b. Write `fix(t, a, b, before)`. Try demotion first: if adding (t, a) keeps the orderings free of cycles, return "demotion". Otherwise try promotion with (b, t) and return "promotion". If both make a cycle, return None. You may copy your consistent function into the cell.

Hint: Add the new pair to a copy of the list and test it with consistent.

In [ ]:
def consistent(before):
    # paste your answer from the last task
    pass

def fix(t, a, b, before):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("fix('T', 'A', 'B', [('A', 'B')])", 'demotion'), ("fix('T', 'A', 'B', [('A', 'B'), ('A', 'T')])", 'promotion'), ("fix('T', 'A', 'B', [('A', 'T'), ('T', 'B')])", None), ("fix('T', 'Start', 'B', [('Start', 'A'), ('A', 'B'), ('Start', 'T')])", 'promotion')])

### Task 3: Count the linearizations of a partial plan

Write `count_orders(steps, before)` that returns how many total orders of the steps respect every pair (a, b) in before.

Hint: Pick any step whose required earlier steps are all placed, then count the orders of the rest. Add up over all such choices.

In [ ]:
def count_orders(steps, before):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("count_orders(['RS', 'RH', 'LS', 'LH'], [('RS', 'RH'), ('LS', 'LH')])", 6), ("count_orders(['A', 'B', 'C', 'D', 'E'], [('A', 'B'), ('B', 'C'), ('D', 'E')])", 10), ("count_orders(['A', 'B', 'C'], [])", 6), ("count_orders(['A', 'B', 'C'], [('A', 'B'), ('B', 'C')])", 1)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Check that orderings have no cycle

Test 3 has the cycle A ≺ B ≺ C ≺ A.

In [ ]:
def consistent(before):
    steps = []
    for a, b in before:
        if a not in steps:
            steps.append(a)
        if b not in steps:
            steps.append(b)
    for s in steps:
        seen = []
        todo = [s]
        while todo:
            x = todo.pop()
            for a, b in before:
                if a == x and b not in seen:
                    seen.append(b)
                    todo.append(b)
        if s in seen:
            return False
    return True

In [ ]:
check([("consistent([('A', 'B'), ('B', 'C')])", True), ("consistent([('A', 'B'), ('B', 'A')])", False), ("consistent([('A', 'B'), ('B', 'C'), ('C', 'A')])", False), ('consistent([])', True)])

### Solution to task 2: Fix a threat by demotion or promotion

Test 3: t is already forced between a and b, so the threat cannot be fixed and the planner must go back. Test 4: demotion would put T before Start, which makes a cycle with Start ≺ T.

In [ ]:
def consistent(before):
    steps = []
    for a, b in before:
        if a not in steps:
            steps.append(a)
        if b not in steps:
            steps.append(b)
    for s in steps:
        seen = []
        todo = [s]
        while todo:
            x = todo.pop()
            for a, b in before:
                if a == x and b not in seen:
                    seen.append(b)
                    todo.append(b)
        if s in seen:
            return False
    return True

def fix(t, a, b, before):
    if consistent(before + [(t, a)]):
        return "demotion"
    if consistent(before + [(b, t)]):
        return "promotion"
    return None

In [ ]:
check([("fix('T', 'A', 'B', [('A', 'B')])", 'demotion'), ("fix('T', 'A', 'B', [('A', 'B'), ('A', 'T')])", 'promotion'), ("fix('T', 'A', 'B', [('A', 'T'), ('T', 'B')])", None), ("fix('T', 'Start', 'B', [('Start', 'A'), ('A', 'B'), ('Start', 'T')])", 'promotion')])

### Solution to task 3: Count the linearizations of a partial plan

Test 2 has chains of 3 and 2 steps: 5! ÷ (3! × 2!) = 10.

In [ ]:
def count_orders(steps, before):
    if len(steps) == 0:
        return 1
    total = 0
    for s in steps:
        ready = True
        for a, b in before:
            if b == s and a in steps:
                ready = False
        if ready:
            rest = []
            for x in steps:
                if x != s:
                    rest.append(x)
            total = total + count_orders(rest, before)
    return total

In [ ]:
check([("count_orders(['RS', 'RH', 'LS', 'LH'], [('RS', 'RH'), ('LS', 'LH')])", 6), ("count_orders(['A', 'B', 'C', 'D', 'E'], [('A', 'B'), ('B', 'C'), ('D', 'E')])", 10), ("count_orders(['A', 'B', 'C'], [])", 6), ("count_orders(['A', 'B', 'C'], [('A', 'B'), ('B', 'C')])", 1)])